# 4 — Scalability on the LB07-bunny benchmark

A scalability study on the **LB07-bunny** max-flow benchmark from the DTU Min-Cut/Max-Flow collection, a large flow network derived from a 3D surface-reconstruction problem. Connected subgraphs of increasing size (from a few hundred up to tens of thousands of vertices) are extracted, and the execution time of the Ancestor Tree is compared against the brute-force approach as the network grows.

In [ ]:
import tarfile
import os

FILE_PATH = "LB07-bunny-sml.tbz2"
EXTRACT_DIR = "LB07-bunny-sml"

# Estrai archivio .tbz2
with tarfile.open(FILE_PATH, "r:bz2") as tar:
    tar.extractall(EXTRACT_DIR)

print("Estratto in:", EXTRACT_DIR)
print("Contenuto:")
for item in os.listdir(EXTRACT_DIR):
    print(" -", item)


Estratto in: LB07-bunny-sml
Contenuto:
 - LB07-bunny-sml.max
 - LB07-bunny-sml.sol


In [ ]:
FILE_PATH = os.path.join(EXTRACT_DIR, "bunny-sml.max")
print("", FILE_PATH)


 LB07-bunny-sml/bunny-sml.max


In [ ]:
import networkx as nx
import math
import time
import random
from collections import deque

# Lettura file DIMACS
def read_dimacs_maxflow(file_path, min_capacity_threshold=0.0):
    Gd = nx.DiGraph()
    s = t = None
    with open(file_path) as f:
        for line in f:
            if not line.strip() or line.startswith("c"):
                continue
            parts = line.split()
            if parts[0] == "n":
                if parts[2] == "s":
                    s = int(parts[1])
                elif parts[2] == "t":
                    t = int(parts[1])
            elif parts[0] == "a":
                u, v, cap = int(parts[1]), int(parts[2]), float(parts[3])
                if cap < min_capacity_threshold:
                    continue
                if Gd.has_edge(u, v):
                    Gd[u][v]["capacity"] += cap
                else:
                    Gd.add_edge(u, v, capacity=cap)
    if s is None or t is None:
        s, t = min(Gd.nodes()), max(Gd.nodes())
    return Gd, s, t


# Conversione a grafo non orientato
def to_undirected_capacitated(Gd):
    Gu = nx.Graph()
    for u, v, data in Gd.edges(data=True):
        c = float(data.get("capacity", 0.0))
        if c <= 0:
            continue
        if Gu.has_edge(u, v):
            Gu[u][v]["capacity"] += c
        else:
            Gu.add_edge(u, v, capacity=c)
    return Gu


# 3️⃣ Capacità totale per costante M
def total_capacity(Gu):
    return sum(float(d.get("capacity", 0.0)) for _, _, d in Gu.edges(data=True))



# 4️⃣ Min-cut vincolato f_st(x,y)
CUT_CALLS = {"count": 0}

def mincut_fst_inplace(Gu, s, t, x, y, M):
    if x == y:
        return (0.0, ({x}, set(Gu.nodes()) - {x}))
    CUT_CALLS["count"] += 2

    # Scenario A: (x,s) e (y,t)
    addedA = []
    for a, b in [(x, s), (y, t)]:
        if a == b:
            continue
        if Gu.has_edge(a, b):
            Gu[a][b]["capacity"] += M
            addedA.append((a, b, M, True))
        else:
            Gu.add_edge(a, b, capacity=M)
            addedA.append((a, b, M, False))
    v1, part1 = nx.minimum_cut(Gu, s, t, capacity="capacity")
    for a, b, cap, existed in addedA:
        if existed:
            Gu[a][b]["capacity"] -= cap
        else:
            Gu.remove_edge(a, b)

    # Scenario B: (x,t) e (y,s)
    addedB = []
    for a, b in [(x, t), (y, s)]:
        if a == b:
            continue
        if Gu.has_edge(a, b):
            Gu[a][b]["capacity"] += M
            addedB.append((a, b, M, True))
        else:
            Gu.add_edge(a, b, capacity=M)
            addedB.append((a, b, M, False))
    v2, part2 = nx.minimum_cut(Gu, s, t, capacity="capacity")
    for a, b, cap, existed in addedB:
        if existed:
            Gu[a][b]["capacity"] -= cap
        else:
            Gu.remove_edge(a, b)

    return (v1, part1) if v1 <= v2 else (v2, part2)



# Struttura dati per Ancestor Tree
class ATNode:
    def __init__(self, members):
        self.members = set(members)
        self.value = None
        self.left = None
        self.right = None
        self.parent = None



# Costruzione Ancestor Tree
def build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42):
    random.seed(seed)
    all_nodes = list(Gu.nodes())
    root = ATNode(all_nodes)
    M = total_capacity(Gu) + 1.0
    seeds = {random.choice(all_nodes)}
    cuts_done = 0
    leaves = [root]

    def pick_unseed(L):
        for z in sorted(L.members):
            if z not in seeds:
                return z
        return None

    def collect_leaf_nodes(node, acc):
        if node.left is None and node.right is None:
            acc.append(node)
            return
        if node.left:
            collect_leaf_nodes(node.left, acc)
        if node.right:
            collect_leaf_nodes(node.right, acc)

    def find_insertion_point(start_node, cut_value):
        x = start_node
        while (
            x.parent is not None
            and x.parent.value is not None
            and cut_value < x.parent.value
        ):
            x = x.parent
        return x

    while cuts_done < len(all_nodes) - 1:
        leaf = next((L for L in leaves if any(z not in seeds for z in L.members)), None)
        if leaf is None:
            break

        p = next((z for z in sorted(leaf.members) if z in seeds), None)
        q = pick_unseed(leaf)
        if p is None or q is None:
            seeds |= set(leaf.members)
            leaves.remove(leaf)
            continue

        cut_value, part = mincut_fst_inplace(Gu, s, t, p, q, M)
        S, T = part
        left_members = {z for z in leaf.members if z in S}
        right_members = leaf.members - left_members

        # Caso (a): inserimento locale
        if left_members and right_members:
            leaf.value = float(cut_value)
            Lnode = ATNode(left_members)
            Rnode = ATNode(right_members)
            Lnode.parent = leaf
            Rnode.parent = leaf
            leaf.left = Lnode
            leaf.right = Rnode
            leaves.remove(leaf)
            leaves += [Lnode, Rnode]
            seeds.add(p)
            seeds.add(q)
            cuts_done += 1
            continue

        # Caso (b): restructuring
        insert_node = find_insertion_point(leaf, float(cut_value))
        A = {z for z in insert_node.members if z in S}
        B = insert_node.members - A
        if not A or not B:
            seeds |= set(leaf.members)
            leaves.remove(leaf)
            continue

        new_node = ATNode(insert_node.members)
        new_node.value = float(cut_value)
        Lnode = ATNode(A)
        Rnode = ATNode(B)
        Lnode.parent = new_node
        Rnode.parent = new_node
        new_node.left = Lnode
        new_node.right = Rnode

        parent = insert_node.parent
        if parent is not None:
            if parent.left is insert_node:
                parent.left = new_node
            else:
                parent.right = new_node
            new_node.parent = parent
        else:
            root = new_node
            new_node.parent = None

        old_leaves = []
        collect_leaf_nodes(insert_node, old_leaves)
        for lf in old_leaves:
            if lf in leaves:
                leaves.remove(lf)
        leaves += [Lnode, Rnode]

        seeds.add(p)
        seeds.add(q)
        cuts_done += 1

    expected = 2 * (len(all_nodes) - 1)
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {expected} attesi")
    return root



# Funzione LCA
def lca_value(root, u, v):
    def path_to_root(x):
        path = []
        def dfs(node):
            if x in node.members:
                path.append(node)
                if node.left and x in node.left.members:
                    dfs(node.left)
                elif node.right and x in node.right.members:
                    dfs(node.right)
        dfs(root)
        return path

    pu, pv = path_to_root(u), path_to_root(v)
    lca = None
    for na, nb in zip(pu, pv):
        if na is nb:
            lca = na
        else:
            break
    while lca and lca.value is None:
        lca = lca.parent
    return None if lca is None else lca.value



# Vitalità
def compute_edge_vitalities(Gu, s, t, root, base_cut_value, warn=True):
    vitality = {}
    bad_edges = []
    for u, v, data in Gu.edges(data=True):
        c_e = float(data.get("capacity", 0.0))
        val_uv = lca_value(root, u, v)
        if val_uv is None:
            continue
        if val_uv < c_e - 1e-9:
            if warn:
                bad_edges.append(((u, v), c_e, val_uv))
            val_uv = c_e
        vitality[(u, v)] = max(0.0, base_cut_value - (val_uv - c_e))

    if bad_edges and warn:
        print(f"[WARN] {len(bad_edges)} archi con LCA < capacity (corretto a c_e). Esempio:")
        for (u, v), ce, vu in bad_edges[:5]:
            print(f"       e=({u},{v})  c_e={ce:.6f}  LCA={vu:.6f}")
    return vitality

In [ ]:
# Esecuzione
if __name__ == "__main__":
    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"

    Gd, s, t = read_dimacs_maxflow(FILE_PATH)
    print(f"Grafo originale: |V|={Gd.number_of_nodes()}, |E|={Gd.number_of_edges()}, s={s}, t={t}")

    # Sottografo bilanciato
    def bfs_subgraph(G, start, max_nodes=550, reverse=False):
        visited = {start}
        queue = [start]
        while queue and len(visited) < max_nodes:
            u = queue.pop(0)
            nbrs = G.predecessors(u) if reverse else G.successors(u)
            for v in nbrs:
                if v not in visited:
                    visited.add(v)
                    queue.append(v)
                    if len(visited) >= max_nodes:
                        break
        return G.subgraph(visited).copy()

    Gs = bfs_subgraph(Gd, s, max_nodes=350)
    Gt = bfs_subgraph(Gd, t, max_nodes=350, reverse=True)
    subset = set(Gs.nodes()) | set(Gt.nodes()) | {s, t}
    Gsub = Gd.subgraph(subset).copy()
    Gu = to_undirected_capacitated(Gsub)

    if not nx.is_connected(Gu):
        for comp in nx.connected_components(Gu):
            if s in comp and t in comp:
                Gu = Gu.subgraph(comp).copy()
                break

    print(f"Sottografo bilanciato: |V|={Gu.number_of_nodes()}, |E|={Gu.number_of_edges()}")

    # Min-cut base
    start = time.time()
    base_value, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s-t: {base_value:.6f} (tempo {time.time()-start:.2f}s)")

    # Costruzione Ancestor Tree
    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_build = time.time() - start
    print(f"Ancestor Tree costruito (tempo {t_build:.2f}s)")
    print("E originali dopo build:", Gu.number_of_edges())

    # Vitalità (Ancestor Tree)
    start = time.time()
    vitality = compute_edge_vitalities(Gu, s, t, root, base_value)
    dt = time.time() - start
    print(f"Vitalità calcolate su {len(vitality)} archi (tempo {dt:.2f}s)")

    # Rimozione arco alla volta
    def brute_force_vitality(Gu, s, t, base_value, tol=1e-9):
        """
        Vitalità brute-force: per ogni arco e=(u,v) rimuovi e, ricomputa min-cut s-t,
        e prendi max(0, base_value - new_cut). Restituisce dict {(u,v): val}.
        """
        vit = {}

        edge_list = list(Gu.edges(data=True))
        for u, v, data in edge_list:
            cap = float(data.get("capacity", 0.0))
            if cap <= tol:
                vit[(u, v)] = 0.0
                continue
            # Rimuovi temporaneamente
            existed = Gu.has_edge(u, v)
            saved_attr = None
            if existed:
                saved_attr = Gu[u][v]
                Gu.remove_edge(u, v)
            try:
                new_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
            except Exception:
                # Se disconnesso o errore, consideriamo cut infinito -> vitalità = base_value
                new_val = float("inf")

            if existed:
                Gu.add_edge(u, v, **saved_attr)

            drop = max(0.0, base_value - new_val if new_val != float("inf") else base_value)
            vit[(u, v)] = drop
        return vit

    start = time.time()
    vitality_bruteforce = brute_force_vitality(Gu, s, t, base_value)
    tb = time.time() - start
    print(f"Brute-force vitalità completato su {len(vitality_bruteforce)} archi (tempo {tb:.2f}s)")

    # Confronto metodi
    def compare_vitalities(v_at, v_bf, tol=1e-6):
        mismatches = []
        common = set(v_at.keys()) & set(v_bf.keys())
        for e in common:
            if abs(v_at[e] - v_bf[e]) > tol:
                mismatches.append((e, v_at[e], v_bf[e], v_at[e]-v_bf[e]))
        only_at = sorted(list(set(v_at.keys()) - set(v_bf.keys())))
        only_bf = sorted(list(set(v_bf.keys()) - set(v_at.keys())))
        return mismatches, only_at, only_bf

    mismatches, only_at, only_bf = compare_vitalities(vitality, vitality_bruteforce, tol=1e-6)
    print("=== Confronto AncestorTree vs BruteForce ===")
    print(f"Archi in comune: {len(set(vitality.keys()) & set(vitality_bruteforce.keys()))}")
    print(f"Discrepanze (>1e-6): {len(mismatches)}")
    if mismatches:
        for k, (e, va, vb, diff) in enumerate(mismatches[:10], 1):
            u, v = e
            print(f"  {k:>2}. {u}--{v}: AT={va:.6f}, BF={vb:.6f}, diff={diff:.6e}")
        if len(mismatches) > 10:
            print(f"  ... altre {len(mismatches)-10} discrepanze non mostrate")
    if only_at:
        print(f"Archi solo in AT: {len(only_at)} (primi 10) -> {only_at[:10]}")
    if only_bf:
        print(f"Archi solo in BF: {len(only_bf)} (primi 10) -> {only_bf[:10]}")

    # --- Top-10 (visualizzazione) basato su AT ---
    top = sorted(vitality.items(), key=lambda kv: -kv[1])[:10]


Grafo originale: |V|=805802, |E|=5040834, s=1, t=2
Sottografo bilanciato: |V|=700, |E|=2112
Min-cut base s-t: 0.000000 (tempo 0.02s)
Min-cut chiamati: 1398 ≈ 1398 attesi
Ancestor Tree costruito (tempo 99.45s)
E originali dopo build: 2112
Vitalità calcolate su 2112 archi (tempo 0.36s)
Brute-force vitalità completato su 2112 archi (tempo 50.33s)
=== Confronto AncestorTree vs BruteForce ===
Archi in comune: 2112
Discrepanze (>1e-6): 0


In [ ]:
# Esecuzione completa (confronto AT vs rimozione archi)
if __name__ == "__main__":
    import numpy as np
    from scipy.stats import kendalltau, spearmanr
    import time

    # Dataset
    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"

    Gd, s, t, neg_count, neg_sum = read_dimacs_maxflow(FILE_PATH) # Updated to accept 5 values
    print(f"Grafo originale: |V|={Gd.number_of_nodes()}, |E|={Gd.number_of_edges()}, s={s}, t={t}")

    # Conversione a non orientato
    Gu_full = to_undirected_capacitated(Gd)

    # Costruzione sottografo connesso e bilanciato
    def grow_from_seed(G, seed, budget, exclude=set()):
        from collections import deque
        seen = set([seed]) | set(exclude)
        out = [seed]
        q = deque([seed])
        while q and len(out) < budget:
            u = q.popleft()
            for v in G.neighbors(u):
                if v not in seen:
                    seen.add(v)
                    out.append(v)
                    q.append(v)
                    if len(out) >= budget:
                        break
        return out

    # Verifica connettività s–t
    if not nx.has_path(Gu_full, s, t):
        raise RuntimeError("⚠️ Il grafo non contiene alcun cammino s–t, impossibile calcolare flusso.")

    spath = nx.shortest_path(Gu_full, s, t)
    TARGET_NODES = 600
    core = set(spath)
    residual = max(TARGET_NODES - len(core), 0)
    quota = residual // 2

    left = grow_from_seed(Gu_full, s, quota, exclude=core)
    right = grow_from_seed(Gu_full, t, quota, exclude=core | set(left))
    subset = core | set(left) | set(right)

    if len(subset) < TARGET_NODES:
        mid = spath[len(spath)//2]
        extra = grow_from_seed(Gu_full, mid, TARGET_NODES - len(subset), exclude=subset)
        subset |= set(extra)

    Gu = Gu_full.subgraph(subset).copy()
    print(f"Sottografo connesso & bilanciato: |V|={Gu.number_of_nodes()}, |E|={Gu.number_of_edges()}")
    print("s–t connessi nel sottografo?", nx.has_path(Gu, s, t))

    # Min-cut base
    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s-t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # Metodo Ancestor Tree
    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val) # Removed unused variable
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # Metodo rimozione archi
    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start
    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # Confronto AT vs Brute-force
    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print(" CONFRONTO METODO RIMOZIONE ARCHI vs ANCESTOR TREE\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}x più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n I due metodi coincidono.")
    else:
        print("\ Piccole differenze numeriche tra AT e metodo di rimozione.")

Grafo originale: |V|=805802, |E|=5040834, s=1, t=2
Sottografo connesso & bilanciato: |V|=599, |E|=1783
s–t connessi nel sottografo? True
Min-cut base s-t: 60.000000 (tempo 0.03s)

Min-cut chiamati: 1196 ≈ 1196 attesi
Ancestor Tree: build=75.69s, vitality=0.31s, totale=76.00s
Metodo rimozione archi: tempo totale 122.34s

🔎 CONFRONTO METODO RIMOZIONE ARCHI vs ANCESTOR TREE

Min-cut chiamati: 1196 ≈ 1196 attesi
Archi confrontati: 1783
Tempo Ancestor Tree: 75.998s (build 75.685s + vitalità 0.313s)
Tempo Metodo rimozione archi: 122.344s
Rapporto velocità (Metodo/AT): 1.6x più lento

Errore medio assoluto (MAE): 0.000000e+00
Errore massimo assoluto: 0.000000e+00
Kendall τ = 1.0000, Spearman ρ = 1.0000
Overlap top-10 archi: 10/10

✅ I due metodi coincidono.


In [ ]:
# Esecuzione completa (confronto AT vs rimozione archi)
if __name__ == "__main__":
    import numpy as np
    from scipy.stats import kendalltau, spearmanr
    import time

    # Dataset
    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"

    Gd, s, t, neg_count, neg_sum = read_dimacs_maxflow(FILE_PATH) # Updated to accept 5 values
    print(f"Grafo originale: |V|={Gd.number_of_nodes()}, |E|={Gd.number_of_edges()}, s={s}, t={t}")

    # Conversione a non orientato
    Gu_full = to_undirected_capacitated(Gd)

    # Costruzione sottografo connesso e bilanciato
    def grow_from_seed(G, seed, budget, exclude=set()):
        from collections import deque
        seen = set([seed]) | set(exclude)
        out = [seed]
        q = deque([seed])
        while q and len(out) < budget:
            u = q.popleft()
            for v in G.neighbors(u):
                if v not in seen:
                    seen.add(v)
                    out.append(v)
                    q.append(v)
                    if len(out) >= budget:
                        break
        return out

    # Verifica connettività s–t
    if not nx.has_path(Gu_full, s, t):
        raise RuntimeError("⚠️ Il grafo non contiene alcun cammino s–t, impossibile calcolare flusso.")

    spath = nx.shortest_path(Gu_full, s, t)
    TARGET_NODES = 1000
    core = set(spath)
    residual = max(TARGET_NODES - len(core), 0)
    quota = residual // 2

    left = grow_from_seed(Gu_full, s, quota, exclude=core)
    right = grow_from_seed(Gu_full, t, quota, exclude=core | set(left))
    subset = core | set(left) | set(right)

    if len(subset) < TARGET_NODES:
        mid = spath[len(spath)//2]
        extra = grow_from_seed(Gu_full, mid, TARGET_NODES - len(subset), exclude=subset)
        subset |= set(extra)

    Gu = Gu_full.subgraph(subset).copy()
    print(f"Sottografo connesso & bilanciato: |V|={Gu.number_of_nodes()}, |E|={Gu.number_of_edges()}")
    print("s–t connessi nel sottografo?", nx.has_path(Gu, s, t))

    # Min-cut base
    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s-t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # Metodo Ancestor Tree
    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val) # Removed unused variable
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # Metodo rimozione archi
    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start
    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # Confronto AT vs Brute-force
    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print(" CONFRONTO METODO RIMOZIONE ARCHI vs ANCESTOR TREE\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}x più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n I due metodi coincidono.")
    else:
        print("\n Piccole differenze numeriche tra AT e metodo di rimozione.")

Grafo originale: |V|=805802, |E|=5040834, s=1, t=2
Sottografo connesso & bilanciato: |V|=999, |E|=3090
s–t connessi nel sottografo? True
Min-cut base s-t: 60.000000 (tempo 0.05s)

Min-cut chiamati: 1996 ≈ 1996 attesi
Ancestor Tree: build=242.44s, vitality=0.85s, totale=243.29s
Metodo rimozione archi: tempo totale 370.08s

🔎 CONFRONTO METODO RIMOZIONE ARCHI vs ANCESTOR TREE

Min-cut chiamati: 1996 ≈ 1996 attesi
Archi confrontati: 3090
Tempo Ancestor Tree: 243.290s (build 242.441s + vitalità 0.849s)
Tempo Metodo rimozione archi: 370.081s
Rapporto velocità (Metodo/AT): 1.5x più lento

Errore medio assoluto (MAE): 1.941748e-02
Errore massimo assoluto: 6.000000e+01
Kendall τ = 0.8655, Spearman ρ = 0.8656
Overlap top-10 archi: 9/10

⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.


In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 200  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")


═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 200, archi: 471
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 0.02s)

Min-cut chiamati: 398 ≈ 398 attesi
Ancestor Tree: build=2.24s, vitality=0.04s, totale=2.28s
Metodo rimozione archi: tempo totale 3.11s

═════════════════════════════════════════════════════
 CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI 
═════════════════════════════════════════════════════

Min-cut chiamati: 398 ≈ 398 attesi
Archi confrontati: 471
Tempo Ancestor Tree: 2.278s (build 2.242s + vitalità 0.036s)
Tempo

In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 600  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")

═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 600, archi: 1604
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 0.03s)

Min-cut chiamati: 1198 ≈ 1198 attesi
Ancestor Tree: build=56.11s, vitality=0.49s, totale=56.60s
Metodo rimozione archi: tempo totale 78.62s

═════════════════════════════════════════════════════
 CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI 
═════════════════════════════════════════════════════

Min-cut chiamati: 1198 ≈ 1198 attesi
Archi confrontati: 1604
Tempo Ancestor Tree: 56.599s (build 56.114s + vitalità 0.

In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr
import sys # Import the sys module

# Increase the recursion depth limit
sys.setrecursionlimit(2000) # You might need to adjust this value depending on the graph size

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 1200  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")

═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 1200, archi: 3322
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 0.04s)

Min-cut chiamati: 2398 ≈ 2398 attesi
Ancestor Tree: build=203.36s, vitality=2.00s, totale=205.36s
Metodo rimozione archi: tempo totale 285.49s

═════════════════════════════════════════════════════
 CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI 
═════════════════════════════════════════════════════

Min-cut chiamati: 2398 ≈ 2398 attesi
Archi confrontati: 3322
Tempo Ancestor Tree: 205.361s (build 203.363s + vital

In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr
import sys # Import the sys module

# Increase the recursion depth limit
sys.setrecursionlimit(4000) # You might need to adjust this value depending on the graph size

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 2500  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")

═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 2500, archi: 6951
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 0.07s)

Min-cut chiamati: 4998 ≈ 4998 attesi
Ancestor Tree: build=1052.28s, vitality=7.98s, totale=1060.26s
Metodo rimozione archi: tempo totale 1511.94s

═════════════════════════════════════════════════════
 CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI 
═════════════════════════════════════════════════════

Min-cut chiamati: 4998 ≈ 4998 attesi
Archi confrontati: 6951
Tempo Ancestor Tree: 1060.258s (build 1052.276s + 

In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr
import sys # Import the sys module

# Increase the recursion depth limit
sys.setrecursionlimit(7000) # You might need to adjust this value depending on the graph size

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 5000  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")

═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 5000, archi: 14168
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 0.12s)

Min-cut chiamati: 9998 ≈ 9998 attesi
Ancestor Tree: build=4164.59s, vitality=34.43s, totale=4199.02s
Metodo rimozione archi: tempo totale 6134.84s

═════════════════════════════════════════════════════
 CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI 
═════════════════════════════════════════════════════

Min-cut chiamati: 9998 ≈ 9998 attesi
Archi confrontati: 14168
Tempo Ancestor Tree: 4199.020s (build 4164.586s

In [ ]:
import networkx as nx
import numpy as np
import time
from scipy.stats import kendalltau, spearmanr
import sys # Import the sys module

# Increase the recursion depth limit
sys.setrecursionlimit(12000) # You might need to adjust this value depending on the graph size

# ============================================================
# 📦 FUNZIONI DI SUPPORTO (devono già esistere nel tuo ambiente)
# ============================================================
# - read_dimacs_maxflow(FILE_PATH)
# - to_undirected_capacitated(Gd)
# - build_ancestor_tree_cheng_hu_with_restructuring(...)
# - compute_edge_vitalities(...)
# - CUT_CALLS dictionary

# ============================================================
# 🚀 ESECUZIONE COMPLETA (Ancestor Tree vs Rimozione Archi)
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 10000  # 🔹 Numero massimo di nodi nel sottografo

    # ========================================================
    # 📥 LETTURA GRAFO E CONTROLLO INFORMAZIONI BASE
    # ========================================================

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    # 🔹 CONTEGGIO DOPO IL CARICAMENTO
    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)

    # Nodi a distanza 1 dal cammino
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))

    subset = core | neighbors_1

    # Se il sottografo supera il limite, riduci mantenendo s,t,core
    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Nodi nel cammino: {len(core)}")
    print(f"🔹 Nodi a distanza 1: {len(neighbors_1)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🌳 METODO ANCESTOR TREE
    # ========================================================

    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(Gu, s, t, seed=42)
    t_at_build = time.time() - start

    start = time.time()
    vitality_at = compute_edge_vitalities(Gu, s, t, root, base_val)
    t_at_vit = time.time() - start
    t_total_at = t_at_build + t_at_vit

    print(f"Ancestor Tree: build={t_at_build:.2f}s, vitality={t_at_vit:.2f}s, totale={t_total_at:.2f}s")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force)
    # ========================================================

    start = time.time()
    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    for u, v, data in edges:
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)
    t_bf_total = time.time() - start

    print(f"Metodo rimozione archi: tempo totale {t_bf_total:.2f}s\n")

    # ========================================================
    # 📈 CONFRONTO TRA I DUE METODI
    # ========================================================

    common_edges = set(vitality_at) & set(vitality_bf)
    bf_vals = np.array([vitality_bf[e] for e in common_edges])
    at_vals = np.array([vitality_at[e] for e in common_edges])

    mae = np.mean(np.abs(bf_vals - at_vals))
    max_err = np.max(np.abs(bf_vals - at_vals))
    tau, _ = kendalltau(bf_vals, at_vals)
    rho, _ = spearmanr(bf_vals, at_vals)

    top_bf = set([e for e, _ in sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]])
    top_at = set([e for e, _ in sorted(vitality_at.items(), key=lambda kv: -kv[1])[:10]])
    overlap = len(top_bf & top_at)

    print("═════════════════════════════════════════════════════")
    print(" CONFRONTO: ANCESTOR TREE vs RIMOZIONE ARCHI ")
    print("═════════════════════════════════════════════════════\n")
    print(f"Min-cut chiamati: {CUT_CALLS['count']} ≈ {2*(Gu.number_of_nodes()-1)} attesi")
    print(f"Archi confrontati: {len(common_edges)}")
    print(f"Tempo Ancestor Tree: {t_total_at:.3f}s (build {t_at_build:.3f}s + vitalità {t_at_vit:.3f}s)")
    print(f"Tempo Metodo rimozione archi: {t_bf_total:.3f}s")
    print(f"Rapporto velocità (Metodo/AT): {t_bf_total / t_total_at:.1f}× più lento\n")
    print(f"Errore medio assoluto (MAE): {mae:.6e}")
    print(f"Errore massimo assoluto: {max_err:.6e}")
    print(f"Kendall τ = {tau:.4f}, Spearman ρ = {rho:.4f}")
    print(f"Overlap top-10 archi: {overlap}/10")

    if mae < 1e-6:
        print("\n✅ I due metodi coincidono perfettamente.")
    else:
        print("\n⚠️ Piccole differenze numeriche tra AT e metodo di rimozione.")

═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Nodi nel cammino: 4
🔹 Nodi a distanza 1: 258352
🔹 Totale nodi nel sottografo: 10000, archi: 28259
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 5.18s)



In [ ]:
import networkx as nx
import numpy as np
import time
import sys

# Aumenta il limite di ricorsione
sys.setrecursionlimit(12000)

# ============================================================
# 🚀 ESECUZIONE: SOLO METODO DI RIMOZIONE ARCHI
# ============================================================

if __name__ == "__main__":

    FILE_PATH = "LB07-bunny-sml/LB07-bunny-sml.max"
    TARGET_NODES = 10000  # 🔹 Numero massimo di nodi nel sottografo

    result = read_dimacs_maxflow(FILE_PATH)
    if len(result) == 3:
        Gd, s, t = result
        neg_count = 0
        neg_sum = 0.0
    else:
        Gd, s, t, neg_count, neg_sum = result

    total_edges = Gd.number_of_edges()
    total_nodes = Gd.number_of_nodes()
    total_cap = sum(abs(data.get("capacity", 0)) for _, _, data in Gd.edges(data=True))
    min_cap = min(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))
    max_cap = max(data.get("capacity", 0) for _, _, data in Gd.edges(data=True))

    print("═════════════════════════════════════════════════════")
    print(f"📊 GRAFO CARICATO: {FILE_PATH}")
    print(f"🔹 Nodi totali: {total_nodes}")
    print(f"🔹 Archi totali: {total_edges}")
    print(f"🔹 Capacità totale: {total_cap:.2f}")
    print(f"🔹 Capacità minima: {min_cap:.2f}")
    print(f"🔹 Capacità massima: {max_cap:.2f}")
    print(f"🔹 Archi con capacità negativa: {neg_count} (somma={neg_sum:.3f})")
    print(f"🔹 Sorgente (s): {s}")
    print(f"🔹 Pozzo (t): {t}")
    print("═════════════════════════════════════════════════════\n")

    # ========================================================
    # 🔄 CONVERSIONE IN GRAFO NON ORIENTATO
    # ========================================================

    Gu_full = to_undirected_capacitated(Gd)

    # ========================================================
    # 🧭 COSTRUZIONE SOTTOGRAFO: NODI A DISTANZA ≤ 1 DALLO SHORTEST PATH
    # ========================================================

    spath = nx.shortest_path(Gu_full, s, t)
    core = set(spath)
    neighbors_1 = set()
    for u in core:
        neighbors_1.update(Gu_full.neighbors(u))
    subset = core | neighbors_1

    if len(subset) > TARGET_NODES:
        essential = {s, t} | core
        extra = list(subset - essential)
        keep_extra = max(0, TARGET_NODES - len(essential))
        subset = essential | set(extra[:keep_extra])

    Gu = Gu_full.subgraph(subset).copy()

    if not nx.has_path(Gu, s, t):
        print("⚠️ s–t disconnessi: ricostruisco sottografo con BFS…")
        Gu = Gu_full.subgraph(core).copy()
        for u in list(core):
            for v in Gu_full.neighbors(u):
                if v not in Gu and len(Gu) < TARGET_NODES:
                    Gu.add_node(v)
                    for w in Gu_full.neighbors(v):
                        if w in Gu:
                            Gu.add_edge(v, w, **Gu_full[v][w])

    print(f"\n🔹 Cammino s–t più corto: lunghezza {len(spath)}")
    print(f"🔹 Totale nodi nel sottografo: {Gu.number_of_nodes()}, archi: {Gu.number_of_edges()}")
    print(f"s–t connessi nel sottografo? {nx.has_path(Gu, s, t)}\n")

    # ========================================================
    # ⚙️ CALCOLO MIN-CUT BASE
    # ========================================================

    start = time.time()
    base_val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
    print(f"Min-cut base s–t: {base_val:.6f} (tempo {time.time()-start:.2f}s)\n")

    # ========================================================
    # 🧱 METODO DI RIMOZIONE ARCHI (Brute-force) + monitoraggio
    # ========================================================

    vitality_bf = {}
    edges = list(Gu.edges(data=True))
    total_edges = len(edges)

    start_total = time.time()
    checkpoint = start_total

    for i, (u, v, data) in enumerate(edges, start=1):
        c = data["capacity"]
        Gu.remove_edge(u, v)
        val, _ = nx.minimum_cut(Gu, s, t, capacity="capacity")
        vitality_bf[(u, v)] = max(0.0, base_val - val)
        Gu.add_edge(u, v, capacity=c)

        # ⏱️ Ogni 2500 archi → stampa tempo parziale
        if i % 1000 == 0 or i == total_edges:
            elapsed = time.time() - checkpoint
            total_elapsed = time.time() - start_total
            perc = 100 * i / total_edges
            print(f"⏳ Avanzamento: {i}/{total_edges} archi ({perc:.1f}%) – "
                  f"{elapsed:.2f}s dall’ultimo checkpoint, {total_elapsed/60:.1f} min totali")
            checkpoint = time.time()

    t_bf_total = time.time() - start_total

    # ========================================================
    # 📊 RISULTATI FINALI
    # ========================================================

    print("═════════════════════════════════════════════════════")
    print("     RISULTATI METODO DI RIMOZIONE ARCHI (Brute-force)")
    print("═════════════════════════════════════════════════════\n")
    print(f"Archi analizzati: {len(vitality_bf)}")
    print(f"Tempo totale: {t_bf_total/60:.2f} minuti\n")

    top_edges = sorted(vitality_bf.items(), key=lambda kv: -kv[1])[:10]
    print("🔝 Top-10 archi per vitalità:")
    for i, (e, v) in enumerate(top_edges, 1):
        print(f"{i:2d}. {e} → {v:.6f}")


═════════════════════════════════════════════════════
📊 GRAFO CARICATO: LB07-bunny-sml/LB07-bunny-sml.max
🔹 Nodi totali: 805802
🔹 Archi totali: 5040834
🔹 Capacità totale: 1688775663.00
🔹 Capacità minima: 1.00
🔹 Capacità massima: 30000.00
🔹 Archi con capacità negativa: 0 (somma=0.000)
🔹 Sorgente (s): 1
🔹 Pozzo (t): 2
═════════════════════════════════════════════════════


🔹 Cammino s–t più corto: lunghezza 4
🔹 Totale nodi nel sottografo: 10000, archi: 28259
s–t connessi nel sottografo? True

Min-cut base s–t: 62.000000 (tempo 1.94s)

⏳ Avanzamento: 1000/28259 archi (3.5%) – 786.38s dall’ultimo checkpoint, 13.1 min totali
⏳ Avanzamento: 2000/28259 archi (7.1%) – 801.24s dall’ultimo checkpoint, 26.5 min totali
⏳ Avanzamento: 3000/28259 archi (10.6%) – 810.50s dall’ultimo checkpoint, 40.0 min totali
⏳ Avanzamento: 4000/28259 archi (14.2%) – 815.25s dall’ultimo checkpoint, 53.6 min totali
⏳ Avanzamento: 5000/28259 archi (17.7%) – 818.33s dall’ultimo checkpoint, 67.2 min totali
⏳ Avanzamento

In [ ]:
import networkx as nx
import random
import time
import numpy as np

def build_dense_graph(n, p):
    """
    Costruisce un grafo orientato molto denso
    con capacità che rendono il min-cut davvero costoso.
    """
    G = nx.DiGraph()
    for u in range(n):
        for v in range(n):
            if u != v and random.random() < p:
                # capacità “alte” → min-cut più costosi
                G.add_edge(u, v, capacity=random.randint(50, 200))
    return G


In [ ]:
def run_experiment(n, p):
    print(f"\n=== Esperimento n={n}, p={p} ===")
    G = build_dense_graph(n, p)

    s, t = 0, n-1

    # mincut base
    base_val, _ = nx.minimum_cut(G, s, t, capacity="capacity")

    # Brute-force
    start = time.time()
    for (u, v, data) in list(G.edges(data=True)):
        cap = data["capacity"]
        G.remove_edge(u, v)
        nx.minimum_cut(G, s, t, capacity="capacity")
        G.add_edge(u, v, capacity=cap)
    bf_time = time.time() - start

    # Ancestor Tree
    CUT_CALLS["count"] = 0
    start = time.time()
    root = build_ancestor_tree_cheng_hu_with_restructuring(G, s, t)
    at_time = time.time() - start

    bf_mincut = len(G.edges())        # BF calls ≈ m
    at_mincut = CUT_CALLS["count"]    # AT calls ≈ 2(n−1)

    print(f"Brute time : {bf_time:.2f}s, mincut calls={bf_mincut}")
    print(f"AT time    : {at_time:.2f}s, mincut calls={at_mincut}")
    print(f"ratio times: {bf_time/at_time:.2f}")
    print(f"ratio m/n  : {(bf_mincut)/(2*n):.2f}")
